In [24]:
import pandas as pd
import numpy as np
import re
import os
import torch
from datasets import Dataset
from transformers import AutoTokenizer

In [25]:
df = pd.read_csv('../data/raw/myanmar_sentiment_simple.csv')
print('Data file loaded')

Data file loaded


In [26]:
df2 =pd.read_csv('../data/raw/myanmar_sentiment_full.csv')

In [27]:
df.columns = df.columns.str.lower()
df.columns

Index(['text', 'sentiment'], dtype='str')

In [28]:
# Method 1: Print one per line
for sentiment in df['sentiment'].str.strip().unique():
    print(f"'{sentiment}'")

'Positive'
'Negative'
'Neutral'
'Anger'
'Fear'
'Sadness'
'Disgust'
'Happiness'
'Joy'
'Love'
'Amusement'
'Enjoyment'
'Admiration'
'Affection'
'Awe'
'Disappointed'
'Surprise'
'Acceptance'
'Adoration'
'Anticipation'
'Bitter'
'Calmness'
'Confusion'
'Excitement'
'Kind'
'Pride'
'Shame'
'Elation'
'Euphoria'
'Contentment'
'Serenity'
'Gratitude'
'Hope'
'Empowerment'
'Compassion'
'Tenderness'
'Arousal'
'Enthusiasm'
'Fulfillment'
'Reverence'
'Despair'
'Grief'
'Loneliness'
'Jealousy'
'Resentment'
'Frustration'
'Boredom'
'Anxiety'
'Intimidation'
'Helplessness'
'Envy'
'Regret'
'Curiosity'
'Indifference'
'Numbness'
'Melancholy'
'Nostalgia'
'Ambivalence'
'Determination'
'Zest'
'Hopeful'
'Proud'
'Grateful'
'Empathetic'
'Compassionate'
'Playful'
'Free-spirited'
'Inspired'
'Confident'
'Bitterness'
'Yearning'
'Fearful'
'Apprehensive'
'Overwhelmed'
'Jealous'
'Devastated'
'Frustrated'
'Envious'
'Dismissive'
'Thrill'
'Bittersweet'
'Overjoyed'
'Inspiration'
'Motivation'
'Contemplation'
'JoyfulReunion'
'Satisf

In [32]:
import pandas as pd
mapping_df = pd.read_csv('../data/annotated/anotate.csv')

mapping_dict = mapping_df.set_index(mapping_df['sentiment'].str.strip())['class'].to_dict()
df['emotion_class'] = df['sentiment'].str.strip().map(mapping_dict)
df['emotion_class'] = df['emotion_class'].fillna('Neutral')

print("=== Final Check ===")
print(f"Null values: {df['emotion_class'].isnull().sum()}")

print("\n=== 6-Class Emotion Distribution ===")
print(df['emotion_class'].value_counts())

print("\n=== Percentages ===")
print(df['emotion_class'].value_counts(normalize=True) * 100)

remaining_unmapped = df[df['emotion_class'] == 'Neutral']['sentiment'].unique()
if len(remaining_unmapped) > 0:
    print(f"\n⚠️ Labels mapped to 'Neutral' (not originally in your CSV mapping):")
    for label in remaining_unmapped[:10]:
        print(f"  - '{label}'")

=== Final Check ===
Null values: 0

=== 6-Class Emotion Distribution ===
emotion_class
Joy         307
Sadness     135
Surprise     71
Neutral      70
Love         61
Anger        58
Fear         30
Name: count, dtype: int64

=== Percentages ===
emotion_class
Joy         41.939891
Sadness     18.442623
Surprise     9.699454
Neutral      9.562842
Love         8.333333
Anger        7.923497
Fear         4.098361
Name: proportion, dtype: float64

⚠️ Labels mapped to 'Neutral' (not originally in your CSV mapping):
  - ' Yearning '
  - ' Dismissive '
  - ' Bittersweet '
  - ' Motivation    '
  - ' Contemplation '
  - ' Reflection    '
  - ' Confidence    '
  - ' Mindfulness   '
  - ' DreamChaser   '
  - ' Elegance '


In [33]:
# View sample of original sentiment vs emotion_class
print("\n=== Sample Mapping ===")
sample_df = df[['sentiment', 'emotion_class']].drop_duplicates().head(30)
print(sample_df.to_string(index=False))


=== Sample Mapping ===
     sentiment emotion_class
    Positive             Joy
    Negative           Anger
    Neutral          Sadness
 Anger                 Anger
 Fear                   Fear
 Sadness             Sadness
 Disgust               Anger
 Happiness               Joy
 Joy                     Joy
 Love                   Love
 Amusement               Joy
 Enjoyment               Joy
 Admiration             Love
 Affection              Love
 Awe                Surprise
 Disappointed        Sadness
 Surprise           Surprise
 Acceptance              Joy
 Adoration              Love
 Anticipation            Joy
 Bitter                Anger
 Calmness                Joy
 Confusion          Surprise
 Excitement              Joy
 Kind                   Love
 Pride                   Joy
 Shame                  Fear
    Confusion       Surprise
   Excitement            Joy
        Shame           Fear


In [34]:
df.columns

Index(['text', 'sentiment', 'emotion_class'], dtype='str')

In [35]:
df['text_length_chars'] = df['text'].astype(str).str.len()
df['text_length_words'] = df['text'].astype(str).str.split().str.len()
print("\n🔤 8. ENCODING CHECK (Zawgyi vs Unicode)...")


def detect_zawgyi(text):
    """Simple heuristic to detect Zawgyi encoding"""
    if not isinstance(text, str):
        return False
    zawgyi_chars = ['ဥ', 'ဦ', 'ဧ', 'ဩ', 'ဪ']
    unicode_chars = ['ဉ', 'ည', 'ဋ', 'ဌ', 'ဍ']
    zawgyi_score = sum(1 for c in zawgyi_chars if c in text)
    unicode_score = sum(1 for c in unicode_chars if c in text)
    return zawgyi_score > unicode_score


df['is_zawgyi'] = df['text'].apply(detect_zawgyi)
zawgyi_count = df['is_zawgyi'].sum()
unicode_count = len(df) - zawgyi_count

print(f"\n   Zawgyi encoded: {zawgyi_count} ({zawgyi_count/len(df)*100:.1f}%)")
print(f"   Unicode encoded: {unicode_count} ({unicode_count/len(df)*100:.1f}%)")

if zawgyi_count > 0:
    print(f"\n   ⚠️  WARNING: Found {zawgyi_count} Zawgyi-encoded texts!")
    print(f"   These need conversion to Unicode before training.")
    print(f"\n   Sample Zawgyi text:")
    zawgyi_sample = df[df['is_zawgyi'] == True]['text'].iloc[0]
    print(f"      {zawgyi_sample[:100]}")
else:
    print(f"\n   ✅ All texts are in Unicode format!")


🔤 8. ENCODING CHECK (Zawgyi vs Unicode)...

   Zawgyi encoded: 6 (0.8%)
   Unicode encoded: 726 (99.2%)

   ⚠️  WARNING: Found 6 Zawgyi-encoded texts!
   These need conversion to Unicode before training.

   Sample Zawgyi text:
      နွေဦးရာသီအတွက် ဥယျာဉ်ကို ပြုပြင်မွမ်းမံဖို့ စီစဉ်နေပါတယ်


In [36]:
# Remove all Zawgyi rows
df = df[df['is_zawgyi'] == False]

# Reset index
df = df.reset_index(drop=True)

print(f"New dataset size: {len(df)} rows")
print(f"Removed {6 - df['is_zawgyi'].sum()} Zawgyi rows")

New dataset size: 726 rows
Removed 6 Zawgyi rows


In [37]:
print("=== PREPROCESSING COMPLETE ===")
print(f"✅ Total samples: {len(df)}")
print(f"✅ 6-class emotions: {df['emotion_class'].nunique()}")
print(f"✅ Unicode encoding: 100%")
print(f"\nEmotion distribution:")
print(df['emotion_class'].value_counts())

=== PREPROCESSING COMPLETE ===
✅ Total samples: 726
✅ 6-class emotions: 7
✅ Unicode encoding: 100%

Emotion distribution:
emotion_class
Joy         303
Sadness     134
Surprise     71
Neutral      69
Love         61
Anger        58
Fear         30
Name: count, dtype: int64


In [38]:
# Define which columns to keep
columns_to_keep = [
    'text',           # Myanmar text content
    'emotion_class'   # 6-class emotion label
]

# Check if these columns exist
available_columns = [col for col in columns_to_keep if col in df.columns]
print(f"Keeping: {available_columns}")

# Remove all other columns
df = df[available_columns]

print(f"\n✅ New dataframe shape: {df.shape}")
print(f"✅ Columns: {df.columns.tolist()}")

Keeping: ['text', 'emotion_class']

✅ New dataframe shape: (726, 2)
✅ Columns: ['text', 'emotion_class']


In [39]:
df.columns

Index(['text', 'emotion_class'], dtype='str')

In [40]:
# Check for duplicate rows
print("=== DUPLICATE CHECK ===")
print(f"Total rows before removal: {len(df)}")

# Check exact duplicates (all columns)
duplicates_all = df.duplicated()
print(f"Exact duplicate rows: {duplicates_all.sum()}")

# Check duplicates based on text column only
duplicates_text = df.duplicated(subset=['text'])
print(f"Duplicate texts: {duplicates_text.sum()}")

# Show percentage
print(f"\nDuplicate percentage: {(duplicates_text.sum() / len(df)) * 100:.2f}%")

=== DUPLICATE CHECK ===
Total rows before removal: 726
Exact duplicate rows: 26
Duplicate texts: 26

Duplicate percentage: 3.58%


In [41]:
# Verify no duplicates remain
print("\n=== FINAL CHECK ===")
print(f"Total rows: {len(df)}")
print(f"Unique texts: {df['text'].nunique()}")
print(f"Duplicate texts remaining: {df.duplicated(subset=['text']).sum()}")

# Check distribution after deduplication
print("\n=== EMOTION DISTRIBUTION AFTER DEDUPLICATION ===")
print(df['emotion_class'].value_counts())
print(f"\nPercentages:")
print(df['emotion_class'].value_counts(normalize=True) * 100)


=== FINAL CHECK ===
Total rows: 726
Unique texts: 700
Duplicate texts remaining: 26

=== EMOTION DISTRIBUTION AFTER DEDUPLICATION ===
emotion_class
Joy         303
Sadness     134
Surprise     71
Neutral      69
Love         61
Anger        58
Fear         30
Name: count, dtype: int64

Percentages:
emotion_class
Joy         41.735537
Sadness     18.457300
Surprise     9.779614
Neutral      9.504132
Love         8.402204
Anger        7.988981
Fear         4.132231
Name: proportion, dtype: float64


In [42]:
df.head(10)

,text,emotion_class
0,ပန်းခြံထဲမှာ လှပတဲ့နေ့လေးတစ်နေ့ကို ပျော်ရွှင်စ...,Joy
1,ဒီမနက် လမ်းအရမ်းပိတ်တယ်,Anger
2,အရမ်းကောင်းတဲ့ လေ့ကျင့်ခန်းတစ်ခု ခုပဲပြီးသွားပ...,Joy
3,လာမယ့် စနေ၊ တနင်္ဂနွေ အပန်းဖြေခရီးအတွက် စိတ်လှ...,Joy
4,ဒီည ညစာအတွက် ဟင်းချက်နည်းအသစ်တစ်ခု စမ်းသပ်ချက်...,Sadness
5,ဘဝရဲ့ သေးငယ်တဲ့အရာလေးတွေအတွက် ကျေးဇူးတင်မိပါတယ်,Joy
6,မိုးရွာတဲ့နေ့တွေက နွေးထွေးတဲ့စောင်တွေနဲ့ ပူပူန...,Joy
7,အသစ်ထွက်ရှိတဲ့ ရုပ်ရှင်က ကြည့်ကိုကြည့်သင့်တဲ့ ...,Joy
8,Timeline မှာ နိုင်ငံရေးဆွေးနွေးမှုတွေ ပူလောင်န...,Anger
9,နွေရာသီရဲ့ ခံစားချက်တွေနဲ့ ကမ်းခြေနေ့ရက်တွေကို...,Sadness


In [43]:
df.to_csv('../data/cleaned/dataset_cleaned.csv', index=False)
print("Saved")

Saved
